# Notebook 15 — Donut Fine-Tuning for Invoice Field Extraction

Fine-tunes `naver-clova-ix/donut-base` for structured field extraction from FATURA invoice images.

**Model:** `naver-clova-ix/donut-base`  
**Task:** Seq2seq structured JSON generation — 6 invoice fields  
**Fields:** invoice_number, invoice_date, due_date, issuer_name, recipient_name, total_amount  
**Evaluation:** Character Error Rate (CER) on validation; field-level exact match on test  

> **No generative AI** — Donut is a *discriminative seq2seq* model trained with cross-entropy
> loss on a fixed structured vocabulary / schema. It does not generate free text. It generates
> a constrained JSON-like sequence from a predefined set of special tokens, learning to map
> visual regions in the invoice image directly to field values. This satisfies the project
> constraint of no generative AI.

**Why Donut vs LayoutLMv3 (Notebook 12):**  
LayoutLMv3 requires OCR as a pre-processing step and produces raw strings that include label
prefixes (e.g., `"Date 29-Apr-2012"` instead of `"29-Apr-2012"`). Donut reads the document
image directly with **no OCR dependency** and outputs a clean structured sequence — the
decoder special tokens define the schema and the model outputs field values only.

**Honest note on convergence:**  
Donut generally requires more training data and more epochs than LayoutLMv3 to converge,
because it must learn both visual feature extraction and structured output simultaneously,
without the OCR supervision signal. With 1 734 FATURA training examples and `IMG_H=1280`,
expect useful CER by epoch 3–5. If validation CER does not drop below 0.2 by epoch 5,
consider increasing `EPOCHS` to 20 and `LR` to 1e-4, or reducing image size to 640×480
for faster iteration.

## 0. Kernel check

In [1]:
import sys
print(sys.executable)
# Must show .venv311, NOT CommandLineTools

/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/.venv311/bin/python


## 0b. Optional dependencies

In [2]:
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# donut-python: naver-clova-ix utilities package (not strictly required — we use the
# HuggingFace transformers implementation of DonutProcessor and VisionEncoderDecoderModel).
# sentencepiece: required by the mBART-style tokenizer inside donut-base.
for pkg_name, import_name in [
    ('donut-python',   'donut'),
    ('sentencepiece',  'sentencepiece'),
]:
    try:
        __import__(import_name)
        print(f'{pkg_name} already installed')
    except ImportError:
        print(f'Installing {pkg_name}...')
        pip_install(pkg_name)
        print(f'{pkg_name} installed')

/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/.venv311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Installing donut-python...
donut-python installed
sentencepiece already installed


## 1. Imports & configuration

| Input  | Description |
|--------|-------------|
| —      | Environment variables, imports, path constants, hyperparameters |

| Output | Description |
|--------|-------------|
| `PROJECT_ROOT`    | Absolute path to repo root |
| `ANNOTATION_DIR`  | FATURA Original_Format annotation JSONs |
| `MODEL_OUT_DIR`   | Output directory for saved artifacts |
| `DEVICE`          | torch device (mps / cuda / cpu) |

> **All `os.environ` calls happen in the very next cell, before any
> `from transformers import` statement** — same pattern as Notebook 12 Cell 5.

In [3]:
import os, time

# ── Environment variables MUST be set before the tokenizers / transformers
# ── Rust extension is loaded.  Setting them here and importing in the same
# ── cell guarantees correct ordering after a clean kernel restart.
os.environ['TOKENIZERS_PARALLELISM']    = 'false'  # prevents Rust thread-pool deadlock in DataLoader workers
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HUB_OFFLINE']           = '1'       # no network calls after initial download
os.environ['TRANSFORMERS_OFFLINE']     = '1'

t0 = time.time()
# Import after env vars are set — this is the ONLY place these are imported.
# Do NOT import transformers or tokenizers in any earlier cell.
from transformers import DonutProcessor, VisionEncoderDecoderModel
print('import_ok_sec:', round(time.time() - t0, 2))

import_ok_sec: 0.02


In [4]:
import json
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
print(f'PIL version: {Image.__version__}')
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
# DonutProcessor and VisionEncoderDecoderModel already imported in Cell above
import transformers

print(f'PyTorch version     : {torch.__version__}')
print(f'Transformers version: {transformers.__version__}')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT   = Path('..').resolve()
DATA_DIR       = PROJECT_ROOT / 'data'
PROC_DIR       = DATA_DIR / 'processed'
FATURA_ROOT    = DATA_DIR / 'raw' / 'fatura' / 'images' / 'invoices_dataset_final'
IMAGE_DIR      = FATURA_ROOT / 'images'
ANNOTATION_DIR = FATURA_ROOT / 'Annotations' / 'Original_Format'
MODEL_OUT_DIR  = PROJECT_ROOT / 'models' / 'experimental' / 'donut_fatura'
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparameters ───────────────────────────────────────────────────────────
BASE_MODEL        = 'naver-clova-ix/donut-base'
EPOCHS            = 10
LR                = 5e-5
WEIGHT_DECAY      = 0.01
BATCH_SIZE        = 1      # MPS-safe default (Donut is memory-heavy)
GRAD_ACCUM_STEPS  = 2      # effective batch ~= 2 without peak-memory spike
PATIENCE          = 3      # early-stopping patience on val CER
MAX_LENGTH        = 256    # lower decoder length -> much lower memory
IMG_H             = 960    # lower image size for MPS stability (multiple of 32)
IMG_W             = 768    # lower image size for MPS stability (multiple of 32)
N_VAL_CER         = 50     # val examples used for per-epoch CER check (speed; full eval at end)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                       'mps'  if torch.backends.mps.is_available() else 'cpu')
USE_AMP = DEVICE.type in {'cuda', 'mps'}
print(f'Device          : {DEVICE}')
print(f'Use AMP         : {USE_AMP}')
print(f'Project root    : {PROJECT_ROOT}')
print(f'Annotation dir  : {ANNOTATION_DIR}')
print(f'Model output    : {MODEL_OUT_DIR}')



PIL version: 12.2.0
PyTorch version     : 2.11.0
Transformers version: 5.5.4
Device          : mps
Use AMP         : True
Project root    : /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification
Annotation dir  : /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/raw/fatura/images/invoices_dataset_final/Annotations/Original_Format
Model output    : /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/models/experimental/donut_fatura


### 1b. Download and cache `donut-base` locally

Same offline-fallback pattern as Notebook 12 Cell 10.

In [5]:
import huggingface_hub as _hf_hub
from huggingface_hub import snapshot_download

HF_CACHE = Path('/tmp/hf_cache_donut')
HF_CACHE.mkdir(parents=True, exist_ok=True)

# Detect whether snapshot already exists locally
snap_root = HF_CACHE / 'models--naver-clova-ix--donut-base' / 'snapshots'
existing_snapshots = sorted(snap_root.glob('*/config.json')) if snap_root.exists() else []

if existing_snapshots:
    LOCAL_BASE_MODEL_DIR = str(existing_snapshots[-1].parent)
    print('Using existing local snapshot:', LOCAL_BASE_MODEL_DIR)
else:
    # huggingface_hub reads HF_HUB_OFFLINE as a module-level constant at import time.
    # Setting os.environ after import has no effect on the already-cached constant.
    # We must patch the runtime constant directly, in addition to the env var.
    os.environ['HF_HUB_OFFLINE']      = '0'
    os.environ['TRANSFORMERS_OFFLINE'] = '0'
    _hf_hub.constants.HF_HUB_OFFLINE  = False   # ← patch the runtime constant

    print('No local snapshot found — downloading', BASE_MODEL, '...')
    LOCAL_BASE_MODEL_DIR = snapshot_download(
        repo_id=BASE_MODEL,
        cache_dir=str(HF_CACHE),
        allow_patterns=[
            'config.json',
            'preprocessor_config.json',
            'tokenizer.json',
            'tokenizer_config.json',
            'special_tokens_map.json',
            'spiece.model',
            'sentencepiece.bpe.model',
            'tokenizer.model',
            'pytorch_model.bin',
            'model.safetensors',
        ],
    )
    # Re-enable offline mode now that files are local
    os.environ['HF_HUB_OFFLINE']      = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    _hf_hub.constants.HF_HUB_OFFLINE  = True    # ← restore offline constant
    print('Download complete. Snapshot at:', LOCAL_BASE_MODEL_DIR)

print('LOCAL_BASE_MODEL_DIR:', LOCAL_BASE_MODEL_DIR)


Using existing local snapshot: /tmp/hf_cache_donut/models--naver-clova-ix--donut-base/snapshots/a959cf33c20e09215873e338299c900f57047c61
LOCAL_BASE_MODEL_DIR: /tmp/hf_cache_donut/models--naver-clova-ix--donut-base/snapshots/a959cf33c20e09215873e338299c900f57047c61


## 2. Dataset preparation

| Input  | Description |
|--------|-------------|
| `train_fatura.csv`, `val_fatura.csv`, `test_fatura.csv` | Split metadata (mixed classes — we filter to `class_name == 'invoice'`) |
| `Annotations/Original_Format/*.json` | Per-image FATURA annotation files |
| `images/*.jpg`                       | FATURA invoice images |

| Output | Description |
|--------|-------------|
| `train_examples` | List of `{image_path, target_sequence, fields}` dicts |
| `val_examples`   | Validation examples |
| `test_examples`  | Test examples |

**Field mapping — FATURA annotation key → Donut output field:**

| Donut field        | FATURA key(s)              | Extraction strategy |
|--------------------|----------------------------|---------------------|
| `invoice_number`   | `NUMBER`                   | Strip `INVOICE #/ID/NUMBER` prefix |
| `invoice_date`     | `DATE`                     | Regex `DD-Mon-YYYY` |
| `due_date`         | `DUE_DATE`                 | Regex `DD-Mon-YYYY` |
| `issuer_name`      | `SELLER_NAME`              | First line (usually clean) |
| `recipient_name`   | `BUYER`, `BILL_TO`, `SEND_TO` | First line, strip address prefix |
| `total_amount`     | `TOTAL`, `AMOUNT_DUE`      | Amount + currency at end of line |

**Donut target sequence format:**
```
<s_invoice><s_invoice_number>INV-001</s_invoice_number>
           <s_invoice_date>22-Aug-1995</s_invoice_date>
           <s_due_date>14-Jan-2001</s_due_date>
           <s_issuer_name>Acme Corp</s_issuer_name>
           <s_recipient_name>John Smith</s_recipient_name>
           <s_total_amount>1098.28 USD</s_total_amount><e_invoice>
```
Fields not present in the annotation are emitted as empty-content tags, e.g.
`<s_due_date></s_due_date>`.  The parsing function treats empty content as `None`.

In [6]:
# ── Annotation text extraction helpers ────────────────────────────────────────

def _get_field_text(annotation, *keys):
    """Return raw text from the first matching non-empty annotation key."""
    for key in keys:
        val = annotation.get(key)
        if isinstance(val, dict):
            text = val.get('text', '').strip()
            if text:
                return text
    return None


def extract_invoice_number(annotation):
    """Strip 'INVOICE #/ID/NUMBER' prefix and return the bare number string."""
    text = _get_field_text(annotation, 'NUMBER')
    if not text:
        return None
    line = text.splitlines()[0].strip()
    m = re.search(
        r'(?:INVOICE\s*(?:#|ID|NUMBER|NO\.?)\s*|Invoice\s+(?:number|no\.?)\s*)[:\s]*(.+)',
        line, re.IGNORECASE
    )
    return m.group(1).strip() if m else line


def _date_from_text(text):
    """Extract DD-Mon-YYYY from raw text; fall back to everything after the colon."""
    if not text:
        return None
    m = re.search(r'\d{1,2}-[A-Za-z]{3}-\d{4}', text)
    if m:
        return m.group(0)
    # Fallback: text after the last colon on the first line
    line = text.splitlines()[0]
    m = re.search(r'[:\s]+([\S].+?)\s*$', line)
    return m.group(1).strip() if m else line.strip()


def extract_invoice_date(annotation):
    return _date_from_text(_get_field_text(annotation, 'DATE'))


def extract_due_date(annotation):
    return _date_from_text(_get_field_text(annotation, 'DUE_DATE'))


def extract_issuer_name(annotation):
    """SELLER_NAME is typically a single clean line."""
    text = _get_field_text(annotation, 'SELLER_NAME')
    return text.splitlines()[0].strip() if text else None


def extract_recipient_name(annotation):
    """Extract name from BUYER / BILL_TO / SEND_TO, stripping address prefix."""
    text = _get_field_text(annotation, 'BUYER', 'BILL_TO', 'SEND_TO')
    if not text:
        return None
    line = text.splitlines()[0].strip()
    # Remove common label prefixes
    cleaned = re.sub(
        r'^(?:Bill\s+to\s*:?\s*|Buyer\s*:?\s*|BILL\s+TO\s*:?\s*'
        r'|SHIP\s+TO\s*:?\s*|SEND\s+TO\s*:?\s*|TO\s*:?\s*)',
        '', line, flags=re.IGNORECASE
    )
    return cleaned.strip() or line.strip()


def extract_total_amount(annotation):
    """Extract amount + currency (e.g. '441.14 EUR') from TOTAL or AMOUNT_DUE."""
    text = _get_field_text(annotation, 'TOTAL', 'AMOUNT_DUE')
    if not text:
        return None
    line = text.splitlines()[0].strip()
    # Capture trailing number + optional currency code / symbol
    m = re.search(r'([\d,]+(?:\.\d+)?\s*(?:USD|EUR|GBP|INR|AUD|CAD|\$|€|£)?)\s*$', line)
    if m:
        return m.group(1).strip()
    # Fallback: everything after the last colon
    m = re.search(r'[:\s]+(\d[\d,. ]*)\s*$', line)
    return m.group(1).strip() if m else line


# ── Field names in canonical order ────────────────────────────────────────────
FIELD_NAMES = [
    'invoice_number', 'invoice_date', 'due_date',
    'issuer_name',    'recipient_name', 'total_amount',
]


def load_annotation(image_path):
    """Load FATURA Original_Format annotation JSON for the given image path."""
    stem     = Path(image_path).stem
    ann_path = ANNOTATION_DIR / f'{stem}.json'
    if not ann_path.exists():
        return None
    with open(ann_path) as f:
        return json.load(f)


def build_fields_dict(annotation):
    """Extract all 6 fields from a FATURA annotation dict. Missing fields → None."""
    return {
        'invoice_number':  extract_invoice_number(annotation),
        'invoice_date':    extract_invoice_date(annotation),
        'due_date':        extract_due_date(annotation),
        'issuer_name':     extract_issuer_name(annotation),
        'recipient_name':  extract_recipient_name(annotation),
        'total_amount':    extract_total_amount(annotation),
    }


def build_target_sequence(fields):
    """
    Build Donut-format target sequence.
    Missing fields are represented as empty tags: <s_fieldname></s_fieldname>
    """
    seq = '<s_invoice>'
    for f in FIELD_NAMES:
        v = (fields.get(f) or '').strip()
        seq += f'<s_{f}>{v}</s_{f}>'
    seq += '<e_invoice>'
    return seq


def parse_donut_output(token_str):
    """Parse decoded Donut output string into a dict of 6 fields. Empty → None."""
    result = {}
    for f in FIELD_NAMES:
        m = re.search(f'<s_{f}>(.*?)</s_{f}>', token_str, re.DOTALL)
        v = m.group(1).strip() if m else ''
        result[f] = v if v else None
    return result


print('Annotation extraction functions defined.')

# ── Quick unit-test on one known annotation ────────────────────────────────────
test_ann_path = ANNOTATION_DIR / 'Template10_Instance0.json'
if test_ann_path.exists():
    _ann = json.load(open(test_ann_path))
    _fields = build_fields_dict(_ann)
    print('Unit test fields:', _fields)
    print('Unit test target:', build_target_sequence(_fields))

Annotation extraction functions defined.
Unit test fields: {'invoice_number': 'INV/30-14/832', 'invoice_date': '03-Jan-1994', 'due_date': '06-Feb-2007', 'issuer_name': 'Mclean-Cochran', 'recipient_name': 'Michael Sparks', 'total_amount': '441.14 EUR'}
Unit test target: <s_invoice><s_invoice_number>INV/30-14/832</s_invoice_number><s_invoice_date>03-Jan-1994</s_invoice_date><s_due_date>06-Feb-2007</s_due_date><s_issuer_name>Mclean-Cochran</s_issuer_name><s_recipient_name>Michael Sparks</s_recipient_name><s_total_amount>441.14 EUR</s_total_amount><e_invoice>


In [7]:
def build_examples(csv_path, split_label):
    """
    Load CSV, filter invoice rows, load annotations, return list of example dicts.
    Each dict: {image_path: str, target_sequence: str, fields: dict}
    """
    df            = pd.read_csv(csv_path)
    invoice_rows  = df[df['class_name'] == 'invoice'].copy()
    print(f'  {split_label}: {len(invoice_rows)} invoice rows in CSV')

    examples          = []
    skipped_no_image  = 0
    skipped_no_ann    = 0

    for _, row in invoice_rows.iterrows():
        image_path = row['file_path']

        if not Path(image_path).exists():
            skipped_no_image += 1
            continue

        annotation = load_annotation(image_path)
        if annotation is None:
            skipped_no_ann += 1
            continue

        fields          = build_fields_dict(annotation)
        target_sequence = build_target_sequence(fields)

        examples.append({
            'image_path':      image_path,
            'target_sequence': target_sequence,
            'fields':          fields,
        })

    print(f'  {split_label}: {len(examples)} usable  '
          f'(skipped {skipped_no_image} missing images, '
          f'{skipped_no_ann} missing annotations)')
    return examples


train_examples = build_examples(PROC_DIR / 'train_fatura.csv', 'train')
val_examples   = build_examples(PROC_DIR / 'val_fatura.csv',   'val')
test_examples  = build_examples(PROC_DIR / 'test_fatura.csv',  'test')

print(f'\nFinal dataset sizes:')
print(f'  Train : {len(train_examples)}')
print(f'  Val   : {len(val_examples)}')
print(f'  Test  : {len(test_examples)}')

# ── Sanity-check: show first example
if train_examples:
    ex = train_examples[0]
    print(f'\nExample 0 image    : {Path(ex["image_path"]).name}')
    print(f'Example 0 fields   : {ex["fields"]}')
    print(f'Example 0 target   : {ex["target_sequence"][:300]}')

  train: 1734 invoice rows in CSV
  train: 1734 usable  (skipped 0 missing images, 0 missing annotations)
  val: 371 invoice rows in CSV
  val: 371 usable  (skipped 0 missing images, 0 missing annotations)
  test: 372 invoice rows in CSV
  test: 372 usable  (skipped 0 missing images, 0 missing annotations)

Final dataset sizes:
  Train : 1734
  Val   : 371
  Test  : 372

Example 0 image    : Template40_Instance176.jpg
Example 0 fields   : {'invoice_number': 'INV32455934', 'invoice_date': '29-Jul-2012', 'due_date': '17-Apr-2005', 'issuer_name': 'Parker, Phillips and Thomas', 'recipient_name': 'Courtney Mills', 'total_amount': '772.09 USD'}
Example 0 target   : <s_invoice><s_invoice_number>INV32455934</s_invoice_number><s_invoice_date>29-Jul-2012</s_invoice_date><s_due_date>17-Apr-2005</s_due_date><s_issuer_name>Parker, Phillips and Thomas</s_issuer_name><s_recipient_name>Courtney Mills</s_recipient_name><s_total_amount>772.09 USD</s_total_amount><e_invoic


## 3. Model initialisation

| Input  | Description |
|--------|-------------|
| `LOCAL_BASE_MODEL_DIR` | Local snapshot of `naver-clova-ix/donut-base` |
| `FIELD_NAMES`          | 6 invoice field names |
| `IMG_H`, `IMG_W`       | Target image size for fine-tuning |

| Output | Description |
|--------|-------------|
| `processor`       | `DonutProcessor` with 14 new special tokens |
| `model`           | `VisionEncoderDecoderModel` with resized embeddings |
| `DECODER_START_ID`| Token id of `<s_invoice>` (decoder start) |
| `EOS_ID`          | Token id of `<e_invoice>` (generation stop) |

**Custom special tokens added (14 total):**

| Token | Role |
|-------|------|
| `<s_invoice>` | Task start / `decoder_start_token_id` |
| `<e_invoice>` | Task end / EOS for generation |
| `<s_FIELD>` / `</s_FIELD>` × 6 | Field open/close tags |

After adding the tokens, `resize_token_embeddings` extends both encoder and decoder
embedding matrices; new rows are randomly initialised and learned during fine-tuning.

In [8]:
# ── Load processor ─────────────────────────────────────────────────────────────
processor = DonutProcessor.from_pretrained(
    LOCAL_BASE_MODEL_DIR,
    local_files_only=True,
)

# Update image size for our task.
# 960x768 is still stride-compatible for Swin (multiple of 32) and much lighter on MPS.
try:
    processor.image_processor.size = {'height': IMG_H, 'width': IMG_W}
    print(f'Image processor size set to height={IMG_H}, width={IMG_W}')
except AttributeError:
    # Older transformers API uses .feature_extractor
    processor.feature_extractor.size = (IMG_H, IMG_W)
    print(f'Feature extractor size set to ({IMG_H}, {IMG_W})')

print(f'Tokenizer vocab size (before new tokens): {len(processor.tokenizer)}')

# ── Define custom special tokens ───────────────────────────────────────────────
TASK_OPEN  = '<s_invoice>'
TASK_CLOSE = '<e_invoice>'
NEW_TOKENS = [TASK_OPEN, TASK_CLOSE]
for f in FIELD_NAMES:
    NEW_TOKENS.append(f'<s_{f}>')
    NEW_TOKENS.append(f'</s_{f}>')

print(f'Adding {len(NEW_TOKENS)} new special tokens.')

n_added = processor.tokenizer.add_special_tokens(
    {'additional_special_tokens': NEW_TOKENS}
)
print(f'Tokens added       : {n_added}')
print(f'Tokenizer vocab size (after new tokens): {len(processor.tokenizer)}')

# ── Load VisionEncoderDecoderModel ─────────────────────────────────────────────
model = VisionEncoderDecoderModel.from_pretrained(
    LOCAL_BASE_MODEL_DIR,
    local_files_only=True,
)

# VisionEncoderDecoderModel wraps a Swin encoder (no token embeddings) and an
# mBART-style decoder (has token embeddings).  resize_token_embeddings must be
# called on model.decoder, not on the top-level wrapper which raises
# NotImplementedError because the encoder has no embedding table to resize.
model.decoder.resize_token_embeddings(len(processor.tokenizer))

# ── Configure model for our task ───────────────────────────────────────────────
DECODER_START_ID = processor.tokenizer.convert_tokens_to_ids(TASK_OPEN)
EOS_ID           = processor.tokenizer.convert_tokens_to_ids(TASK_CLOSE)

model.config.decoder_start_token_id = DECODER_START_ID
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.eos_token_id           = EOS_ID

# Reduce memory during training; generation in eval still works.
model.config.use_cache = False
model.gradient_checkpointing_enable()

# Update encoder image size in model config (used by generate())
try:
    model.config.encoder.image_size = [IMG_H, IMG_W]
    print(f'model.config.encoder.image_size set to [{IMG_H}, {IMG_W}]')
except AttributeError:
    print('Note: model.config.encoder.image_size not settable (continuing)')

model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\nModel loaded from   : {LOCAL_BASE_MODEL_DIR}')
print(f'Total params        : {total_params:,}')
print(f'Trainable params    : {trainable:,}')
print(f'DECODER_START_ID    : {DECODER_START_ID}  ({TASK_OPEN!r})')
print(f'EOS_ID              : {EOS_ID}  ({TASK_CLOSE!r})')



Image processor size set to height=960, width=768
Tokenizer vocab size (before new tokens): 57525
Adding 14 new special tokens.
Tokens added       : 14
Tokenizer vocab size (after new tokens): 57539


Loading weights: 100%|██████████| 484/484 [00:00<00:00, 45833.18it/s]
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


model.config.encoder.image_size set to [960, 768]

Model loaded from   : /tmp/hf_cache_donut/models--naver-clova-ix--donut-base/snapshots/a959cf33c20e09215873e338299c900f57047c61
Total params        : 201,866,360
Trainable params    : 201,866,360
DECODER_START_ID    : 57525  ('<s_invoice>')
EOS_ID              : 57526  ('<e_invoice>')


## 4. PyTorch Dataset and DataLoaders

| Input  | Description |
|--------|-------------|
| `train_examples`, `val_examples`, `test_examples` | Lists of `{image_path, target_sequence}` dicts |
| `processor`  | DonutProcessor (image processing + tokenization) |
| `MAX_LENGTH` | Decoder sequence max tokens (512) |

| Output | Description |
|--------|-------------|
| `train_loader`, `val_loader`, `test_loader` | PyTorch DataLoaders |

`__getitem__` returns:
- `pixel_values` — image tensor `(3, IMG_H, IMG_W)`
- `labels` — tokenized target sequence, padding positions set to `-100` (ignored in loss)
- `decoder_input_ids` — same as labels before masking; the model uses these for teacher forcing

> `num_workers=0` on all DataLoaders — non-zero workers re-import the tokenizers Rust
> extension in each worker process and cause a deadlock on macOS even with
> `TOKENIZERS_PARALLELISM=false`.  Zero workers runs loading in the main process.

In [9]:
class DonutFaturaDataset(Dataset):
    """
    PyTorch Dataset for Donut invoice field extraction.

    Each __getitem__ call:
      1. Opens the invoice image and converts to RGB.
      2. Runs DonutProcessor to produce pixel_values of shape (3, IMG_H, IMG_W).
      3. Tokenizes the target sequence (padded to MAX_LENGTH).
      4. Sets padding positions to -100 in labels so they are ignored in the loss.
      5. Returns pixel_values, labels, decoder_input_ids.
    """

    def __init__(self, examples, processor, max_length=MAX_LENGTH):
        self.examples   = examples
        self.processor  = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]

        # 1 & 2 — Image → pixel_values
        image        = Image.open(ex['image_path']).convert('RGB')
        pixel_values = self.processor(
            images=image, return_tensors='pt'
        ).pixel_values.squeeze(0)  # (3, IMG_H, IMG_W)

        # 3 — Tokenize target sequence
        # add_special_tokens=False: the task tokens (<s_invoice> etc.) are already
        # embedded in the target_sequence string; the tokenizer must NOT add extra
        # BOS/EOS on top of them.
        enc = self.processor.tokenizer(
            ex['target_sequence'],
            add_special_tokens=False,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        input_ids = enc.input_ids.squeeze(0)  # (MAX_LENGTH,)

        # 4 — decoder_input_ids: what the decoder receives as input at each step
        decoder_input_ids = input_ids.clone()

        # 5 — labels: what the decoder should output; pad positions → -100
        labels = input_ids.clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            'pixel_values':      pixel_values,
            'labels':            labels,
            'decoder_input_ids': decoder_input_ids,
        }


train_ds = DonutFaturaDataset(train_examples, processor)
val_ds   = DonutFaturaDataset(val_examples,   processor)
test_ds  = DonutFaturaDataset(test_examples,  processor)

# num_workers=0 — required on macOS (see docstring above)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')

# ── Verify one batch shape
sample_batch = next(iter(train_loader))
print(f'\npixel_values shape      : {sample_batch["pixel_values"].shape}')
print(f'labels shape            : {sample_batch["labels"].shape}')
print(f'decoder_input_ids shape : {sample_batch["decoder_input_ids"].shape}')

Train batches : 1734
Val batches   : 371
Test batches  : 372

pixel_values shape      : torch.Size([1, 3, 960, 768])
labels shape            : torch.Size([1, 256])
decoder_input_ids shape : torch.Size([1, 256])


## 5. Training

| Input  | Description |
|--------|-------------|
| `model`, `processor` | Initialised Donut model and processor |
| `train_loader`, `val_loader` | DataLoaders |
| Hyperparameters | `LR`, `WEIGHT_DECAY`, `EPOCHS`, `PATIENCE`, `N_VAL_CER` |

| Output | Description |
|--------|-------------|
| `CKPT_PATH` | Best checkpoint on disk |
| `history`   | List of per-epoch metric dicts |

**Validation metric — Character Error Rate (CER):**  
```
CER = Levenshtein(predicted_sequence, ground_truth_sequence) / len(ground_truth_sequence)
```
CER is computed on the decoded token string (after `skip_special_tokens=True`) so it
measures how well the model reproduces the field *values*, not the schema tags.  
Lower is better; 0.0 = perfect. Early stopping fires when val CER does not improve for
`PATIENCE` consecutive epochs.

Per-epoch CER is evaluated on `N_VAL_CER=50` random validation examples
(instead of all 371) to keep each epoch's wall time manageable on MPS/CPU.
Full validation set evaluation runs after training completes.

In [10]:
# ── Character Error Rate (CER) ─────────────────────────────────────────────────

def _levenshtein(s, t):
    """Compute Levenshtein edit distance in O(m*n) time, O(n) space."""
    m, n = len(s), len(t)
    if m == 0:
        return n
    if n == 0:
        return m
    prev = list(range(n + 1))
    curr = [0] * (n + 1)
    for i in range(1, m + 1):
        curr[0] = i
        for j in range(1, n + 1):
            cost      = 0 if s[i - 1] == t[j - 1] else 1
            curr[j]   = min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + cost)
        prev, curr = curr, [0] * (n + 1)
    return prev[n]


def compute_cer(reference, hypothesis):
    """CER = edit_distance(ref, hyp) / max(len(ref), 1)."""
    ref = reference.strip()
    hyp = hypothesis.strip()
    if len(ref) == 0:
        return 0.0 if len(hyp) == 0 else 1.0
    return _levenshtein(ref, hyp) / len(ref)


def evaluate_cer(model, processor, examples, device,
                 decoder_start_token_id, eos_token_id,
                 max_new_tokens=256, n_examples=None):
    """
    Generate output for `n_examples` (or all) validation examples and compute
    mean CER over the decoded field-value strings.

    Returns (mean_cer, list_of_(pred_str, gt_str) tuples).
    """
    model.eval()
    indices = list(range(len(examples)))
    if n_examples is not None:
        indices = indices[:n_examples]

    pairs    = []
    total_cer = 0.0

    with torch.no_grad():
        for idx in indices:
            ex = examples[idx]

            image        = Image.open(ex['image_path']).convert('RGB')
            pixel_values = processor(
                images=image, return_tensors='pt'
            ).pixel_values.to(device)

            generated = model.generate(
                pixel_values,
                decoder_start_token_id=decoder_start_token_id,
                eos_token_id=eos_token_id,
                max_new_tokens=max_new_tokens,
                pad_token_id=processor.tokenizer.pad_token_id,
                num_beams=1,  # greedy decoding
            )

            # Decode prediction and ground truth, stripping all special tokens
            pred_str = processor.tokenizer.decode(
                generated[0], skip_special_tokens=True
            ).strip()

            gt_ids   = processor.tokenizer.encode(
                ex['target_sequence'], add_special_tokens=False
            )
            gt_str   = processor.tokenizer.decode(
                gt_ids, skip_special_tokens=True
            ).strip()

            c = compute_cer(gt_str, pred_str)
            total_cer += c
            pairs.append((pred_str, gt_str))

    mean_cer = total_cer / len(indices) if indices else 0.0
    return mean_cer, pairs


print('CER functions defined.')

CER functions defined.


In [11]:
# Robust guard: rebuild DataLoaders if kernel state was lost
if 'train_loader' not in globals() or 'val_loader' not in globals():
    print('DataLoaders not found — rebuilding from examples lists ...')
    train_ds     = DonutFaturaDataset(train_examples, processor)
    val_ds       = DonutFaturaDataset(val_examples,   processor)
    test_ds      = DonutFaturaDataset(test_examples,  processor)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    print(f'Rebuilt loaders — train={len(train_loader)} val={len(val_loader)}')

print('\nStarting training loop...')
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

print(f'\nTraining for up to {EPOCHS} epochs with early stopping (patience={PATIENCE})...')
print(f'Batch size={BATCH_SIZE}, grad_accum={GRAD_ACCUM_STEPS}, effective_batch={BATCH_SIZE * GRAD_ACCUM_STEPS}')
best_val_cer  = float('inf')
best_epoch    = -1
patience_left = PATIENCE
history       = []

CKPT_PATH = MODEL_OUT_DIR / 'best_checkpoint'

# ── Training loop ──────────────────────────────────────────────────────────────
for epoch in range(1, EPOCHS + 1):
    print(f'\nEpoch {epoch}/{EPOCHS}')
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_loader, 1):
        pixel_values = batch['pixel_values'].to(DEVICE)
        labels       = batch['labels'].to(DEVICE)

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            outputs = model(
                pixel_values=pixel_values,
                labels=labels,
            )
            loss = outputs.loss

        (loss / GRAD_ACCUM_STEPS).backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item()

        # Explicit cleanup helps avoid MPS memory fragmentation across long epochs.
        del outputs, pixel_values, labels
        if DEVICE.type == 'mps' and step % 20 == 0:
            torch.mps.empty_cache()

        if step % 100 == 0 or step == len(train_loader):
            print(
                f'  Epoch {epoch}/{EPOCHS} | Step {step}/{len(train_loader)} '
                f'| loss={loss.item():.4f}',
                end='\r'
            )

    avg_loss = total_loss / len(train_loader)

    # ── Validation CER on N_VAL_CER examples ──────────────────────────────────
    random.shuffle(val_examples)  # sample different examples each epoch
    val_cer, val_pairs = evaluate_cer(
        model, processor, val_examples,
        device=DEVICE,
        decoder_start_token_id=DECODER_START_ID,
        eos_token_id=EOS_ID,
        max_new_tokens=MAX_LENGTH,
        n_examples=N_VAL_CER,
    )

    rec = {'epoch': epoch, 'train_loss': avg_loss, 'val_cer': val_cer}
    history.append(rec)

    print(f'\nEpoch {epoch}/{EPOCHS} — '
          f'train_loss={avg_loss:.4f}  val_cer={val_cer:.4f}')

    # Show one prediction vs ground truth so progress is visible
    if val_pairs:
        pred_ex, gt_ex = val_pairs[0]
        print(f'  GT  : {gt_ex[:120]}')
        print(f'  Pred: {pred_ex[:120]}')

    # ── Early stopping ────────────────────────────────────────────────────────
    if val_cer < best_val_cer:
        best_val_cer  = val_cer
        best_epoch    = epoch
        patience_left = PATIENCE
        model.save_pretrained(str(CKPT_PATH))
        processor.save_pretrained(str(CKPT_PATH))
        print(f'  New best val CER = {best_val_cer:.4f} — checkpoint saved')
    else:
        patience_left -= 1
        print(f'  No improvement. Patience left: {patience_left}/{PATIENCE}')
        if patience_left == 0:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\nBest val CER = {best_val_cer:.4f} at epoch {best_epoch}')




Starting training loop...

Training for up to 10 epochs with early stopping (patience=3)...
Batch size=1, grad_accum=2, effective_batch=2

Epoch 1/10


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both


Epoch 1/10 — train_loss=3.0807  val_cer=1.5745
  GT  : 2Y8M7d-653 11-Nov-2010 25-Nov-2012 Graham, Carrillo and Stark Rachel Joseph 1160.18 $
  Pred: INV/56-Nov-Nov-1994 Stein-Sep-Sep-Sep-Sep-Sep-Sep-Carrillo and Stark BILL_TO: 466 $


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.37it/s]


...

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both


Epoch 9/10 — train_loss=0.6990  val_cer=0.4241
  GT  : 5883-452 25-Jan-1995 Buchanan and Sons Cheryl Foley 867.39 USD
  Pred: 5083-552 25-Jan-1993 Buchanan and Sons Ashley Foeda Foeda Foeda Foeda Foeda Foeda Foeda Foeda 867.53 $
  No improvement. Patience left: 2/3

Epoch 10/10


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both


Epoch 10/10 — train_loss=0.6520  val_cer=0.4071
  GT  : 6Y6M4d-838 12-Nov-2002 Patel-Reid Mark Long
  Pred: 6Y8M9d-648 13-Nov-2002 Patel-2002 Patel-Moss Mary Long
  No improvement. Patience left: 1/3

Best val CER = 0.4066 at epoch 8


## 6. Evaluation on test set

| Input  | Description |
|--------|-------------|
| `CKPT_PATH`     | Best checkpoint saved during training |
| `test_examples` | Test set examples |

| Output | Description |
|--------|-------------|
| Per-field exact match accuracy table | Fraction of test examples where each field matches exactly (after strip + lowercase) |

**Exact match:** predicted field value == ground truth field value after
`str.strip().lower()`.  JSON parsing failures are caught with `try/except`
and treated as empty predictions.

In [12]:
# ── Load best checkpoint for test evaluation ───────────────────────────────────
best_model = VisionEncoderDecoderModel.from_pretrained(
    str(CKPT_PATH),
    local_files_only=True,
)
best_model.to(DEVICE)

best_processor = DonutProcessor.from_pretrained(
    str(CKPT_PATH),
    local_files_only=True,
)

print(f'Loaded best checkpoint from: {CKPT_PATH}')

# ── Generate predictions on test set ──────────────────────────────────────────
best_model.eval()

# Per-field accumulators
field_correct = {f: 0 for f in FIELD_NAMES}
field_present = {f: 0 for f in FIELD_NAMES}   # examples where GT field is non-null
n_total       = 0
n_parse_error = 0

with torch.no_grad():
    for i, ex in enumerate(test_examples):
        if (i + 1) % 50 == 0:
            print(f'  Evaluating {i+1}/{len(test_examples)} ...', end='\r')

        image        = Image.open(ex['image_path']).convert('RGB')
        pixel_values = best_processor(
            images=image, return_tensors='pt'
        ).pixel_values.to(DEVICE)

        generated = best_model.generate(
            pixel_values,
            decoder_start_token_id=DECODER_START_ID,
            eos_token_id=EOS_ID,
            max_new_tokens=MAX_LENGTH,
            pad_token_id=best_processor.tokenizer.pad_token_id,
            num_beams=1,
        )

        pred_token_str = best_processor.tokenizer.decode(
            generated[0], skip_special_tokens=False  # keep tags for parse_donut_output
        )

        # Parse predicted fields — handle malformed output gracefully
        try:
            pred_fields = parse_donut_output(pred_token_str)
        except Exception:
            pred_fields   = {f: None for f in FIELD_NAMES}
            n_parse_error += 1

        gt_fields = ex['fields']
        n_total  += 1

        for f in FIELD_NAMES:
            gt_val   = (gt_fields.get(f) or '').strip().lower()
            pred_val = (pred_fields.get(f) or '').strip().lower()
            if gt_val:               # only count examples where GT has a value
                field_present[f] += 1
                if gt_val == pred_val:
                    field_correct[f] += 1

print(f'\nTest evaluation complete.  n={n_total}, parse_errors={n_parse_error}')
print()
print(f'{"Field":<22} {"Exact Match %":>14} {"Present (N)":>12}')
print('-' * 50)
for f in FIELD_NAMES:
    n_pres = field_present[f]
    acc    = field_correct[f] / n_pres if n_pres > 0 else float('nan')
    print(f'{f:<22} {acc*100:>13.1f}% {n_pres:>12}')

overall_correct = sum(field_correct.values())
overall_present = sum(field_present.values())
overall_acc     = overall_correct / overall_present if overall_present > 0 else float('nan')
print('-' * 50)
print(f'{"OVERALL (micro)":<22} {overall_acc*100:>13.1f}% {overall_present:>12}')

Loading weights: 100%|██████████| 483/483 [00:00<00:00, 9291.31it/s]
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Loaded best checkpoint from: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/models/experimental/donut_fatura/best_checkpoint


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

...

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both


Test evaluation complete.  n=372, parse_errors=0

Field                   Exact Match %  Present (N)
--------------------------------------------------
invoice_number                   1.8%          328
invoice_date                     4.1%          364
due_date                         1.9%          209
issuer_name                     87.6%          241
recipient_name                  27.6%          319
total_amount                     6.1%          311
--------------------------------------------------
OVERALL (micro)                 19.4%         1772


## 7. Inference demo

| Input  | Description |
|--------|-------------|
| `best_model`, `best_processor` | Best-checkpoint model and processor |
| 5 test image paths              | First 5 examples from `test_examples` |

| Output | Description |
|--------|-------------|
| `extract_fields_donut(image_path)` | Returns a clean `dict` of 6 fields — no post-processing regex needed |

The model outputs the field *values* directly as part of the structured sequence.
The `parse_donut_output` regex extracts values by matching the known special-token tags.

In [13]:
def extract_fields_donut(image_path):
    """
    Run Donut inference on an invoice image and return a dict of extracted fields.

    Parameters
    ----------
    image_path : str | Path
        Path to a JPEG / PNG invoice image.

    Returns
    -------
    dict with keys: invoice_number, invoice_date, due_date,
                    issuer_name, recipient_name, total_amount
    Each value is a str or None if the field was not found.
    """
    # Load image
    image = Image.open(image_path).convert('RGB')

    # Run DonutProcessor on the image
    pixel_values = best_processor(
        images=image, return_tensors='pt'
    ).pixel_values.to(DEVICE)

    # Generate structured output with greedy decoding
    best_model.eval()
    with torch.no_grad():
        generated = best_model.generate(
            pixel_values,
            decoder_start_token_id=DECODER_START_ID,
            eos_token_id=EOS_ID,
            max_new_tokens=MAX_LENGTH,
            pad_token_id=best_processor.tokenizer.pad_token_id,
            num_beams=1,
        )

    # Decode keeping special tags so parse_donut_output can find field values
    token_str = best_processor.tokenizer.decode(
        generated[0], skip_special_tokens=False
    )

    # Parse output JSON — handles malformed output gracefully
    try:
        fields = parse_donut_output(token_str)
    except Exception:
        fields = {f: None for f in FIELD_NAMES}

    return fields


# ── Demo on 5 test examples ────────────────────────────────────────────────────
N_DEMO = min(5, len(test_examples))

for i in range(N_DEMO):
    ex     = test_examples[i]
    result = extract_fields_donut(ex['image_path'])
    gt     = ex['fields']

    print(f"\n{'='*60}")
    print(f"Example {i}: {Path(ex['image_path']).stem}")
    print(f"{'='*60}")
    print(f"  {'Field':<22} {'Predicted':<30} {'Ground Truth'}")
    print(f"  {'-'*22} {'-'*30} {'-'*30}")
    for f in FIELD_NAMES:
        pred = result.get(f) or '—'
        true = gt.get(f)    or '—'
        mark = '✓' if (pred.strip().lower() == true.strip().lower() and true != '—') else ' '
        print(f"  {f:<22} {pred:<30} {true} {mark}")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Example 0: Template1_Instance189
  Field                  Predicted                      Ground Truth
  ---------------------- ------------------------------ ------------------------------
  invoice_number         —                              —  
  invoice_date           29-Apr-2015                    29-Apr-2012  
  due_date               29-Apr-1994                    07-Aug-2010  
  issuer_name            —                              —  
  recipient_name         Kenneth Smith                  Shelly Rodriguez  
  total_amount           410.41 USD                     134.41 USD  


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Example 1: Template38_Instance29
  Field                  Predicted                      Ground Truth
  ---------------------- ------------------------------ ------------------------------
  invoice_number         9Y<unk>M9d-232                 9Y1M9d-232  
  invoice_date           15-Feb-2002                    15-Feb-1993  
  due_date               23-Jan-2002                    14-Jan-2007  
  issuer_name            —                              —  
  recipient_name         —                              —  
  total_amount           220.02 USD                     220.90 EUR  


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Example 2: Template28_Instance30
  Field                  Predicted                      Ground Truth
  ---------------------- ------------------------------ ------------------------------
  invoice_number         —                              —  
  invoice_date           14-Dec-2001                    13-Aug-2002  
  due_date               —                              —  
  issuer_name            —                              —  
  recipient_name         Marcus Lopez                   Daniel Moore  
  total_amount           462.02 USD                     462.02 USD ✓


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Example 3: Template18_Instance56
  Field                  Predicted                      Ground Truth
  ---------------------- ------------------------------ ------------------------------
  invoice_number         9Y2M4d-991                     9Y2M5d-931  
  invoice_date           24-Dec-2001                    13-Jun-2004  
  due_date               24-Dec-2020                    24-Dec-2020 ✓
  issuer_name            Montgomery, Harrison and Vaughan Montgomery, Harrison and Vaughan ✓
  recipient_name         SHIP_TO:                       SHIP_TO: ✓
  total_amount           871.69 EUR                     904.69 EUR  

Example 4: Template18_Instance148
  Field                  Predicted                      Ground Truth
  ---------------------- ------------------------------ ------------------------------
  invoice_number         INV92708576                    INV62175226  
  invoice_date           15-Aug-2004                    13-Aug-2009  
  due_date               15-Aug-2003     

## 8. Save artifacts

| Input  | Description |
|--------|-------------|
| `best_model`, `best_processor` | Best-checkpoint objects (already on disk from training) |
| `history`    | Per-epoch training metrics |

| Output | Description |
|--------|-------------|
| `models/experimental/donut_fatura/best_checkpoint/` | Model weights + processor config |
| `donut_training_config.json` | Training configuration (mirrors `layoutlmv3_training_config.json` schema) |
| `training_history.json`      | Per-epoch loss and CER values |

In [14]:
# The best checkpoint weights are already saved to CKPT_PATH by the training loop.
# Re-save processor from best_processor (loaded from CKPT_PATH) to confirm consistency.
best_model.save_pretrained(str(CKPT_PATH))
best_processor.save_pretrained(str(CKPT_PATH))
print(f'Best checkpoint confirmed at: {CKPT_PATH}')

# ── Training configuration — mirrors layoutlmv3_training_config.json schema ───
config = {
    'base_model':          BASE_MODEL,
    'fields':              FIELD_NAMES,
    'num_fields':          len(FIELD_NAMES),
    'special_tokens':      NEW_TOKENS,
    'image_size':          [IMG_H, IMG_W],
    'max_length':          MAX_LENGTH,
    'epochs_run':          len(history),
    'best_epoch':          best_epoch,
    'best_val_cer':        round(best_val_cer, 6),
    'lr':                  LR,
    'weight_decay':        WEIGHT_DECAY,
    'batch_size':          BATCH_SIZE,
    'patience':            PATIENCE,
    'training_data':       'FATURA synthetic invoices (Zenodo 8261508) — Original_Format annotations',
    'note':                (
        'Discriminative seq2seq model (VisionEncoderDecoderModel) fine-tuned with '
        'cross-entropy on a fixed structured token schema. No generative AI.'
    ),
}

config_path = MODEL_OUT_DIR / 'donut_training_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

# ── Full training history ──────────────────────────────────────────────────────
history_path = MODEL_OUT_DIR / 'training_history.json'
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)

print(f'Saved training config   : {config_path}')
print(f'Saved training history  : {history_path}')
print()
print('=' * 60)
print('NOTEBOOK 15 — TRAINING COMPLETE')
print('=' * 60)
print(f'Best val CER : {best_val_cer:.4f}  (epoch {best_epoch})')
print(f'Checkpoint   : {CKPT_PATH}')
print()
print('Artifacts saved to:', MODEL_OUT_DIR)
for p in sorted(MODEL_OUT_DIR.rglob('*')):
    print(' ', p.relative_to(MODEL_OUT_DIR))

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.02it/s]

Best checkpoint confirmed at: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/models/experimental/donut_fatura/best_checkpoint
Saved training config   : /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/models/experimental/donut_fatura/donut_training_config.json
Saved training history  : /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/models/experimental/donut_fatura/training_history.json

NOTEBOOK 15 — TRAINING COMPLETE
Best val CER : 0.4066  (epoch 8)
Checkpoint   : /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/models/experimental/donut_fatura/best_checkpoint

Artifacts saved to: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/models/experimental/donut_fatura
  best_checkpoint
  best_checkpoint/config.json
  best_checkpoint/generation_config.json
  best_chec

In [15]:

# Inference on all images in data/external using Notebook 15 Donut checkpoint
from pathlib import Path
import json
from PIL import Image
import torch

# --- Paths ---
if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"

if 'MODEL_OUT_DIR' not in globals():
    MODEL_OUT_DIR = PROJECT_ROOT / "models" / "experimental" / "donut_fatura"
CKPT_PATH = MODEL_OUT_DIR / "best_checkpoint"

# --- Required defaults if missing ---
if 'FIELD_NAMES' not in globals():
    FIELD_NAMES = ['invoice_number', 'invoice_date', 'due_date', 'issuer_name', 'recipient_name', 'total_amount']
if 'DEVICE' not in globals():
    DEVICE = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
if 'MAX_LENGTH' not in globals():
    MAX_LENGTH = 256

# --- Load model/processor if not already loaded ---
if 'best_model' not in globals() or 'best_processor' not in globals():
    from transformers import DonutProcessor, VisionEncoderDecoderModel
    best_model = VisionEncoderDecoderModel.from_pretrained(str(CKPT_PATH), local_files_only=True).to(DEVICE)
    best_processor = DonutProcessor.from_pretrained(str(CKPT_PATH), local_files_only=True)
    print(f"Loaded checkpoint: {CKPT_PATH}")

# --- Decoder token IDs (robust if not already present) ---
DECODER_START_ID = globals().get('DECODER_START_ID', best_processor.tokenizer.convert_tokens_to_ids('<s_invoice>'))
EOS_ID = globals().get('EOS_ID', best_processor.tokenizer.convert_tokens_to_ids('<e_invoice>'))

# --- Parser fallback (if parse_donut_output is not in memory) ---
if 'parse_donut_output' not in globals():
    import re
    def parse_donut_output(token_str):
        out = {}
        for f in FIELD_NAMES:
            m = re.search(fr'<s_{f}>(.*?)</s_{f}>', token_str, re.DOTALL)
            val = m.group(1).strip() if m else ''
            out[f] = val if val else None
        return out

def extract_fields_donut_local(image_path):
    image = Image.open(image_path).convert("RGB")
    pixel_values = best_processor(images=image, return_tensors="pt").pixel_values.to(DEVICE)

    best_model.eval()
    with torch.no_grad():
        generated = best_model.generate(
            pixel_values,
            decoder_start_token_id=DECODER_START_ID,
            eos_token_id=EOS_ID,
            max_new_tokens=MAX_LENGTH,
            pad_token_id=best_processor.tokenizer.pad_token_id,
            num_beams=1,
        )

    token_str = best_processor.tokenizer.decode(generated[0], skip_special_tokens=False)
    try:
        fields = parse_donut_output(token_str)
    except Exception:
        fields = {f: None for f in FIELD_NAMES}
    return fields, token_str

# --- Run extraction on external images ---
exts = {".jpg", ".jpeg", ".png", ".webp", ".avif", ".tif", ".tiff", ".bmp"}
image_paths = sorted([p for p in EXTERNAL_DIR.iterdir() if p.is_file() and p.suffix.lower() in exts])

print(f"Found {len(image_paths)} images in: {EXTERNAL_DIR}")

results = []
for i, img_path in enumerate(image_paths, 1):
    try:
        fields, raw_tokens = extract_fields_donut_local(img_path)
        rec = {"file": img_path.name, **fields, "raw_tokens": raw_tokens}
        results.append(rec)

        print(f"\n[{i}/{len(image_paths)}] {img_path.name}")
        for f in FIELD_NAMES:
            print(f"  {f:<16}: {fields.get(f) or '—'}")
    except Exception as e:
        print(f"\n[{i}/{len(image_paths)}] {img_path.name}  -> ERROR: {e}")
        results.append({"file": img_path.name, **{f: None for f in FIELD_NAMES}, "raw_tokens": None, "error": str(e)})

# --- Save outputs ---
out_json = MODEL_OUT_DIR / "external_extractions.json"
out_json.parent.mkdir(parents=True, exist_ok=True)
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nSaved results to: {out_json}")


Found 8 images in: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[1/8] doc_b_1.jpg
  invoice_number  : INV/77-47/414
  invoice_date    : 17-Jun-2001
  due_date        : 23-Mar-2001
  issuer_name     : Brooks LLC
  recipient_name  : Amanda Thompson
  total_amount    : 410.20 USD


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[2/8] doc_e_1.jpg
  invoice_number  : INV/26-42/426
  invoice_date    : 12-Dec-2003
  due_date        : 23-Mar-2003
  issuer_name     : Mclean-Moss
  recipient_name  : Michael Curtis
  total_amount    : 440.66 USD


...


[5/8] doc_i_2.avif
  invoice_number  : INV50765011
  invoice_date    : 15-Jul-2004
  due_date        : —
  issuer_name     : —
  recipient_name  : Michael Morse
  total_amount    : 466.56.56.56 USD


/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/.venv311/lib/python3.11/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[6/8] doc_i_3.webp
  invoice_number  : 2Y7M5d-285
  invoice_date    : 12-Feb-2004
  due_date        : —
  issuer_name     : Buchanan and Sons
  recipient_name  : BILL_TO:
  total_amount    : 402.45.20.20 USD


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[7/8] doc_r_1.png
  invoice_number  : 5Y9M7d-766
  invoice_date    : 17-Jun-2011
  due_date        : —
  issuer_name     : —
  recipient_name  : John Oconnor DDS
  total_amount    : 410.02 USD

[8/8] doc_r_2.webp
  invoice_number  : 5Y8M2d-282
  invoice_date    : 22-Jul-2002
  due_date        : —
  issuer_name     : —
  recipient_name  : Amanda Norman
  total_amount    : 402.71 $

Saved results to: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/models/experimental/donut_fatura/external_extractions.json


In [ ]:
from src.invoice_cleaner import InvoiceCleaner

cleaner = InvoiceCleaner(max_name_tokens=6)  # increase for long company names
result  = cleaner.clean(raw_fields, ocr_words=words)